# Day 12: Project 1 — Data Cleaning & Feature Engineering

**Dataset:** UCI Diabetes 130-US Hospitals (from Day 11)
**Goal:** Handle missing, group ICD-9 codes, engineer features, save cleaned data

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Load raw data
df = pd.read_csv('data/diabetes_raw.csv')
print(f"Raw shape: {df.shape}")

Raw shape: (101766, 50)


In [10]:
# 1. HANDLE MISSING VALUES
print("=== Missing values before cleaning ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

# Drop columns with >80% missing
df = df.drop(columns=['max_glu_serum', 'A1Cresult'])

# weight: impute with 'Unknown' category
df['weight'] = df['weight'].fillna('Unknown')

# race: impute with mode
df['race'] = df['race'].fillna(df['race'].mode()[0])

# payer_code: impute with mode
df['payer_code'] = df['payer_code'].fillna(df['payer_code'].mode()[0])

# medical_specialty: impute with 'Missing'
df['medical_specialty'] = df['medical_specialty'].fillna('Missing')

print("\n=== Missing values after cleaning ===")
print(df.isnull().sum().sum())

=== Missing values before cleaning ===
max_glu_serum    94.75
A1Cresult        83.28
dtype: float64

=== Missing values after cleaning ===
0


In [11]:
# 2. GROUP ICD-9 CODES INTO CLINICAL CATEGORIES
# Simple mapping: first 3 digits → major category
def icd9_to_category(code):
    if pd.isna(code) or code == '?' or code == '':
        return 'Missing'
    code = str(code).strip()
    # V codes (supplementary)
    if code.startswith('V'):
        return 'Supplementary'
    # E codes (external causes)
    if code.startswith('E'):
        return 'External'
    # Numeric codes
    try:
        num = int(float(code[:3]))
        if 1 <= num <= 139:
            return 'Infectious'
        elif 140 <= num <= 239:
            return 'Neoplasms'
        elif 240 <= num <= 279:
            return 'Endocrine'
        elif 280 <= num <= 289:
            return 'Blood'
        elif 290 <= num <= 319:
            return 'Mental'
        elif 320 <= num <= 389:
            return 'Nervous'
        elif 390 <= num <= 459:
            return 'Circulatory'
        elif 460 <= num <= 519:
            return 'Respiratory'
        elif 520 <= num <= 579:
            return 'Digestive'
        elif 580 <= num <= 629:
            return 'Genitourinary'
        elif 630 <= num <= 679:
            return 'Pregnancy'
        elif 680 <= num <= 709:
            return 'Skin'
        elif 710 <= num <= 739:
            return 'Musculoskeletal'
        elif 740 <= num <= 759:
            return 'Congenital'
        elif 760 <= num <= 779:
            return 'Perinatal'
        elif 780 <= num <= 799:
            return 'Symptoms'
        elif 800 <= num <= 999:
            return 'Injury'
        else:
            return 'Other'
    except:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_cat'] = df[col].apply(icd9_to_category)

print("diag_1 categories:")
print(df['diag_1_cat'].value_counts())
print("\ndiag_2 categories:")
print(df['diag_2_cat'].value_counts())
print("\ndiag_3 categories:")
print(df['diag_3_cat'].value_counts())

diag_1 categories:
diag_1_cat
Circulatory        30336
Endocrine          11459
Respiratory        10407
Digestive           9208
Symptoms            7636
Injury              6974
Genitourinary       5078
Musculoskeletal     4957
Neoplasms           3433
Infectious          2768
Skin                2530
Mental              2262
Supplementary       1644
Nervous             1211
Blood               1103
Pregnancy            687
Congenital            51
Missing               21
External               1
Name: count, dtype: int64

diag_2 categories:
diag_2_cat
Circulatory        31365
Endocrine          21017
Respiratory        10251
Genitourinary       7987
Symptoms            4632
Digestive           3962
Skin                3596
Blood               2926
Mental              2657
Neoplasms           2547
Injury              2428
Infectious          1931
Supplementary       1805
Musculoskeletal     1764
Nervous             1286
External             731
Pregnancy            415
Missing      

In [12]:
# 3. FEATURE ENGINEERING
# Prior admissions (proxy for comorbidity burden)
df['prior_admissions'] = df['number_inpatient'] + df['number_emergency'] + df['number_outpatient']

# Medication changes: count 'Up' + 'Down' across drug columns
drug_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
           'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
           'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
           'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
           'glipizide-metformin', 'glimepiride-pioglitazone',
           'metformin-rosiglitazone', 'metformin-pioglitazone']

df['med_change_count'] = 0
for col in drug_cols:
    df['med_change_count'] += (df[col].isin(['Up', 'Down'])).astype(int)

# Length of stay category
df['los_category'] = pd.cut(df['time_in_hospital'], 
                             bins=[0, 3, 7, 14, 100], 
                             labels=['Short (1-3d)', 'Medium (4-7d)', 'Long (8-14d)', 'Very Long (14+d)'])

# Number of diagnoses category
df['num_diagnoses_cat'] = pd.cut(df['number_diagnoses'],
                                  bins=[0, 5, 8, 12, 20],
                                  labels=['Few (1-5)', 'Moderate (6-8)', 'Many (9-12)', 'Complex (13+)'])

# Age midpoint (for potential numeric use)
age_map = {'[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
           '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
           '[80-90)': 85, '[90-100)': 95}
df['age_midpoint'] = df['age'].map(age_map)

print("New features:")
print(f"  prior_admissions: mean={df['prior_admissions'].mean():.2f}, max={df['prior_admissions'].max()}")
print(f"  med_change_count: mean={df['med_change_count'].mean():.2f}, max={df['med_change_count'].max()}")
print(f"  los_category distribution:\n{df['los_category'].value_counts()}")

New features:
  prior_admissions: mean=1.20, max=80
  med_change_count: mean=0.29, max=4
  los_category distribution:
los_category
Short (1-3d)        49188
Medium (4-7d)       37288
Long (8-14d)        15290
Very Long (14+d)        0
Name: count, dtype: int64


In [13]:
# 4. CLEAN '?' VALUES IN CATEGORICAL COLUMNS
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if '?' in df[col].values:
        df[col] = df[col].replace('?', 'Unknown')

# Also clean 'Unknown/Invalid' in gender
df['gender'] = df['gender'].replace('Unknown/Invalid', 'Unknown')

print("Cleaned '?' values")

Cleaned '?' values


In [14]:
# 5. TARGET VARIABLE
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

print(f"Target distribution: {df['readmitted_binary'].value_counts().to_dict()}")
print(f"Readmission rate: {df['readmitted_binary'].mean():.4f}")

Target distribution: {0: 90409, 1: 11357}
Readmission rate: 0.1116


In [15]:
# 6. SAVE CLEANED DATASET
output_path = 'data/diabetes_clean.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned data saved to {output_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Cleaned data saved to data/diabetes_clean.csv
Shape: (101766, 57)
Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat', 'prior_admissions', 'med_change_count', 'los_category', 'num_diagnoses_cat', 'age_midpoint', 'r